## imports:

In [16]:
%run given.ipynb
import datetime as dt
import re
from types import SimpleNamespace


Salon backend ready: 6 seed bookings. NOW = 2026-10-19 10:00 (Monday)
22 public scenarios loaded.


### cancellation fee cents:

In [17]:
def cancellation_fee_cents(price_cents, start, now):
    """Fee in cents; raise ValueError if the appointment has already started."""
    remaining = start - now
    if remaining <= dt.timedelta(0):
        raise ValueError('the appointment has already started')
    if remaining >= dt.timedelta(hours=24):
        return 0
    if remaining >= dt.timedelta(hours=2):
        return price_cents // 2
    return price_cents

# class SalonAssistant:
#     def __init__(self, model, backend, clock):
#         self.model, self.backend, self.clock = model, backend, clock
#         self.sessions, self.audit = {}, []

#     def open_session(self, session_id, customer_id):
#         raise NotImplementedError('open_session')

#     def handle(self, session_id, user_text):
#         """Return the reply text for the customer."""
#         raise NotImplementedError('handle')

### check cancellation fee cents:

In [18]:
n = dt.datetime(2026, 10, 19, 10, 0)
H = lambda h, m=0: n + dt.timedelta(hours=h, minutes=m)

assert cancellation_fee_cents(3500, H(24), n) == 0   
assert cancellation_fee_cents(3500, H(23, 59), n) == 1750
assert cancellation_fee_cents(3501, H(5), n) == 1750    
assert cancellation_fee_cents(3500, H(2), n) == 1750     
assert cancellation_fee_cents(3500, H(1, 59), n) == 3500

for bad in (n, H(-1)):
    try:
        cancellation_fee_cents(3500, bad, n)
        raise SystemExit("it should be value eror !")
    except ValueError:
        pass
print("fee OK")

fee OK


### interpert reply :

In [19]:
_YES = (
    r"(?:yes|yeah|yep|yup|y|ok|okay|sure|absolutely|definitely|perfect|great|alright|fine|"
    r"confirm|confirmed|correct|go ahead|that works(?: for me)?|(?:that )?sounds good|"
    r"do it|please do|go for it|book it)"
)
_YES_TAIL = r"(?: (?:yes|please|thanks|thank you|do it|do|go ahead|perfect|great|sure|ok|okay))*"
_NO = r"(?:no|nope|nah|stop|not yet|never mind|nevermind|don't|dont|do not|please don't|please dont)"
_NO_TAIL = r"(?: (?:thanks|thank you|please|it|that|book it|cancel it|do it|now))*"
YES_RE = re.compile(rf"^{_YES}{_YES_TAIL}$")
NO_RE  = re.compile(rf"^{_NO}{_NO_TAIL}$")

def interpret_reply(text):
    """Return 'yes', 'no' or 'unclear'. Conservative: when in doubt, 'unclear'."""
    if not isinstance(text, str) or '?' in text:
        return 'unclear'

    s = text.lower().replace('’', "'")
    s = re.sub(r"[^a-z0-9' ]+", ' ', s)
    s = ' '.join(s.split()) 

    if YES_RE.match(s):
        return 'yes'
    if NO_RE.match(s):
        return 'no'
    return 'unclear'


### tool schema:

In [20]:
_SERVICE_IDS = ['cut', 'beard', 'wash_style', 'color', 'highlights']
_STYLIST_IDS = ['mara', 'jon', 'lea']
_START_PATTERN = r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}$'

def _tool(name, description, properties=None, required=None):
    return {'type': 'function', 'function': {
        'name': name, 'description': description,
        'parameters': {'type': 'object', 'properties': properties or {},
                       'required': required or [], 'additionalProperties': False}}}

_SERVICE = {'type': 'string', 'enum': _SERVICE_IDS, 'description': 'Service id'}
_STYLIST = {'type': 'string', 'enum': _STYLIST_IDS, 'description': 'Stylist id'}
_BOOKING = {'type': 'string', 'pattern': r'^B-\d+$', 'description': 'Booking id such as B-2'}
_START = {'type': 'string', 'pattern': _START_PATTERN,
          'description': "Local time 'YYYY-MM-DD HH:MM' on a 30-minute mark"}

TOOL_SCHEMAS = [
    _tool('list_services', 'List the salon services with durations and prices.'),
    _tool('find_slots', 'Find free appointment slots for a service on one day.',
          {'service_id': _SERVICE,
           'date': {'type': 'string', 'pattern': r'^\d{4}-\d{2}-\d{2}$', 'description': "Date 'YYYY-MM-DD'"},
           'stylist_id': _STYLIST}, ['service_id', 'date']),
    _tool('get_my_bookings', "List the logged-in customer's own bookings."),
    _tool('book_appointment', 'PROPOSE a booking. Nothing is booked until the customer confirms in their next message.',
          {'service_id': _SERVICE, 'stylist_id': _STYLIST, 'start': _START},
          ['service_id', 'stylist_id', 'start']),
    _tool('reschedule_appointment', "PROPOSE moving one of the customer's bookings to a new start time (confirmation needed).",
          {'booking_id': _BOOKING, 'new_start': _START}, ['booking_id', 'new_start']),
    _tool('cancel_appointment', "PROPOSE cancelling one of the customer's bookings (confirmation needed; a fee may apply).",
          {'booking_id': _BOOKING}, ['booking_id']),
    _tool('handoff_to_human', 'Pass the case to a human team member.',
          {'reason': {'type': 'string', 'enum': ['customer_request','policy_exception', 'tool_failures', 'other'],
                      'description': 'Why a human is needed'}}, ['reason']),
]

### check p22 & p05:

In [21]:
sc_fee_table(None); print("p22 fee table ok")
sc_reply_table(None); print("p05 reply table ok")

sc_schemas(SimpleNamespace(model=SimpleNamespace(tools_seen=TOOL_SCHEMAS)))
print("p21 schemas ok ")

p22 fee table ok
p05 reply table ok
p21 schemas ok 
